# 4. Method Comparison & Object Detection Benchmark
### Comprehensive Evaluation of HOG vs SIFT vs FREAK

This notebook presents a direct, head-to-head comparison between the three OpenCV feature methods:
1. **HOG (Histogram of Oriented Gradients)**: Dense gradient orientation representation.
2. **SIFT (Scale-Invariant Feature Transform)**: Sparse multi-scale floating-point keypoint descriptors.
3. **FREAK (Fast Retina Keypoint)**: Bio-inspired binary foveal descriptor.



In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Load benchmark results generated by evaluate_all.py
summary_path = Path("benchmark_summary.json")
if not summary_path.exists():
    from evaluate_all import run_benchmark
    run_benchmark()

with open(summary_path, "r") as f:
    results = json.load(f)

methods = list(results.keys())
accuracies = [results[m]["Accuracy (%)"] for m in methods]
latencies = [results[m]["Latency (ms/img)"] for m in methods]
fps_vals = [results[m]["Throughput (FPS)"] for m in methods]
sizes = [results[m]["Model Size (MB)"] for m in methods]

print("=" * 80)
print(f"{'Method':<8} | {'Accuracy':<12} | {'Latency (ms)':<14} | {'FPS':<10} | {'Model Size':<10}")
print("-" * 80)
for m in methods:
    print(f"{m:<8} | {results[m]['Accuracy (%)']:>10.2f}% | {results[m]['Latency (ms/img)']:>12.2f} ms | {results[m]['Throughput (FPS)']:>10.1f} | {results[m]['Model Size (MB)']:>8.2f} MB")
print("=" * 80)



## Visual Comparison Charts


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Accuracy
colors = ['#2b5c8f', '#6c4b8b', '#d95f02']
axes[0].bar(methods, accuracies, color=colors)
axes[0].set_title("Test Accuracy (%)", fontsize=13)
axes[0].set_ylim(0, 100)
for i, v in enumerate(accuracies):
    axes[0].text(i, v + 2, f"{v:.1f}%", ha='center', fontweight='bold')

# 2. Latency / Inference Time
axes[1].bar(methods, latencies, color=colors)
axes[1].set_title("Inference Latency (ms / image)", fontsize=13)
for i, v in enumerate(latencies):
    axes[1].text(i, v + 0.5, f"{v:.2f} ms", ha='center', fontweight='bold')

# 3. Model Size
axes[2].bar(methods, sizes, color=colors)
axes[2].set_title("Model File Footprint (MB)", fontsize=13)
for i, v in enumerate(sizes):
    axes[2].text(i, v + 1, f"{v:.2f} MB", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()



## Visualizing Object Detection Demos (SIFT vs FREAK)


In [ ]:
import cv2

sift_demo = Path("visualizations/sift_detection_demo.jpg")
freak_demo = Path("visualizations/freak_detection_demo.jpg")

if sift_demo.exists() and freak_demo.exists():
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    im_s = cv2.cvtColor(cv2.imread(str(sift_demo)), cv2.COLOR_BGR2RGB)
    im_f = cv2.cvtColor(cv2.imread(str(freak_demo)), cv2.COLOR_BGR2RGB)

    axes[0].imshow(im_s)
    axes[0].set_title("SIFT Object Detection & Homography Bounding Box", fontsize=12)
    axes[0].axis("off")

    axes[1].imshow(im_f)
    axes[1].set_title("FREAK Binary Object Detection & Homography Bounding Box", fontsize=12)
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()



## Architectural Summary & Decision Matrix

| Dimension | HOG | SIFT | FREAK |
| :--- | :--- | :--- | :--- |
| **Descriptor Type** | Dense gradient orientations (8,100 floats) | Sparse keypoints (128 floats / point) | Sparse keypoints (64 bytes / point) |
| **Keypoint Detector** | None (dense grid) | Built-in DoG (Difference of Gaussians) | External (FAST, BRISK, AGAST) |
| **Distance Metric** | L2 / Linear Hyperplane | L2 Norm | Hamming Distance (bitwise XOR) |
| **Classification Accuracy**| **84.29%** (Best on global shape) | 70.00% (BoVW) | 71.43% (BoVW) |
| **Inference Speed** | **0.73 ms** (~1,370 FPS) | 35.34 ms (~28 FPS) | **2.93 ms** (~340 FPS, 12x faster than SIFT!) |
| **Model Size** | 39.38 MB | 0.21 MB | **0.17 MB** |
| **Invariance** | Illumination, local edge gradients | Scale, Rotation, Illumination | Scale, Rotation, Illumination |
| **Best Use Case** | Whole-image category classification & pedestrian/rigid object sliding window | High-precision instance recognition under severe scale/rotation | Real-time / mobile / embedded object detection & SLAM tracking |

